# 03 — Modélisation, GridSearchCV & tracking MLflow

Ce notebook couvre **Milestone 1 & 4**.

- Setup MLflow
- Pipelines anti data leakage (preprocessing + SMOTE + modèle)
- GridSearchCV sur **score métier**
- Logging des résultats et artefacts (ROC, confusion matrix, modèle)

In [1]:
from pathlib import Path
import time
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, f1_score

import sys, os
sys.path.insert(0, os.path.abspath(".."))  # monte d'un niveau (../)
from src.config import PATHS
from src.data import load_parquet
from src.metrics import optimal_threshold_cost, get_proba
from src.models import make_cv, gridsearch_dummy, gridsearch_lgbm_smote, gridsearch_xgb_smote, gridsearch_logreg_smote, gridsearch_rf_smote, run_gridsearch

from src.mlflow_utils import init_mlflow, track_run
import mlflow

## 1) Charger dataset préparé

In [2]:
df = load_parquet(PATHS.data_processed / "train_fe.parquet")

assert "TARGET" in df.columns, "Le dataset doit contenir TARGET (uniquement train)."
X = df.drop(columns=["TARGET"])
y = df["TARGET"].astype(int)

X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
print(X_train.shape, X_valid.shape)
print(set(X.dtypes))
print(X.select_dtypes(include='object').columns.tolist())



(79999, 125) (20000, 125)
{dtype('bool'), dtype('O'), dtype('int64'), dtype('float64')}
['NAME_CONTRACT_TYPE', 'CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'NAME_TYPE_SUITE', 'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE', 'OCCUPATION_TYPE', 'WEEKDAY_APPR_PROCESS_START', 'ORGANIZATION_TYPE', 'FONDKAPREMONT_MODE', 'HOUSETYPE_MODE', 'WALLSMATERIAL_MODE', 'EMERGENCYSTATE_MODE']


## 2) Init MLflow

In [3]:
from datetime import datetime

init_mlflow()

mlflow.set_experiment("credit_scoring")
notebook_run_id = datetime.now().strftime("%Y%m%d_%H%M%S")


C:\apps\anaconda3\Lib\site-packages\mlflow\tracking\_tracking_service\utils.py:177: FutureWarning: The filesystem tracking backend (e.g., './mlruns') will be deprecated in February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://github.com/mlflow/mlflow/issues/18534 for more details and migration guidance.
  return FileStore(store_uri, store_uri)
2026/05/11 18:59:48 INFO mlflow.tracking.fluent: Experiment with name 'credit_scoring' does not exist. Creating a new experiment.


In [4]:
cv = make_cv(2)

In [5]:
from src.models import build_preprocessor

print(X.shape, set(X.dtypes))

pre = build_preprocessor(X)
Xt = pre.fit_transform(X)
print(Xt.shape, Xt.dtype)
print(type(Xt))

(99999, 125) {dtype('bool'), dtype('O'), dtype('int64'), dtype('float64')}
(99999, 256) float64
<class 'numpy.ndarray'>


## 3) LightGBM

In [6]:
pipe, param_grid = gridsearch_lgbm_smote(X_train, y_train, cv)
t0 = time.time()
gs = run_gridsearch(pipe, param_grid, X_train, y_train, cv)
fit_time = time.time() - t0

best_model_lightGBM = gs.best_estimator_
t1 = time.time()
y_proba = get_proba(best_model_lightGBM, X_valid)
predict_time = time.time() - t1

threshold_info = optimal_threshold_cost(y_valid.values, y_proba)
y_pred = (y_proba >= threshold_info["threshold"]).astype(int)

cv_results_df = pd.DataFrame(gs.cv_results_)

extra_metrics_lightGBM = {
    "business_cost_test": threshold_info["cost"],
    "business_threshold_test": threshold_info["threshold"],
    "business_score_cv": gs.best_score_,
    "AUC_test": roc_auc_score(y_valid, y_proba),
    "mean_cv_threshold": gs.cv_results_["mean_test_threshold"][gs.best_index_],
    "fit_time": fit_time,
    "predict_time": predict_time
}


Fitting 2 folds for each of 8 candidates, totalling 16 fits
[LightGBM] [Info] Number of positive: 73525, number of negative: 73525
[LightGBM] [Debug] Dataset::GetMultiBinFromSparseFeatures: sparse rate 0.920723
[LightGBM] [Debug] Dataset::GetMultiBinFromAllFeatures: sparse rate 0.659486
[LightGBM] [Debug] init for col-wise cost 0.091817 seconds, init for row-wise cost 0.153614 seconds
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.109005 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Debug] Using Sparse Multi-Val Bin
[LightGBM] [Info] Total Bins 54600
[LightGBM] [Info] Number of data points in the train set: 147050, number of used features: 238
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Debug] Trained a tree with leaves = 31 and depth = 10
[LightGBM] [Debug] Trained a tree with leaves = 31 and depth = 10

C:\apps\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


### Résultat

In [7]:

gs.best_params_, gs.best_score_, extra_metrics_lightGBM

({'model__colsample_bytree': 0.8,
  'model__learning_rate': 0.05,
  'model__max_depth': 10,
  'model__min_child_samples': 60,
  'model__n_estimators': 400,
  'model__num_leaves': 31,
  'model__subsample': 0.8},
 np.float64(-21595.0),
 {'business_cost_test': 10519.0,
  'business_threshold_test': 0.10999999999999999,
  'business_score_cv': np.float64(-21595.0),
  'AUC_test': 0.7570200907367388,
  'mean_cv_threshold': np.float64(0.0875),
  'fit_time': 219.51464748382568,
  'predict_time': 0.4377923011779785})

### MLFlow tracking

In [8]:
run_name = "lightgbm"

# model_type = f"{run_name}_threshold_cost_optimized"

# mlflow.set_tag("algo", run_name)
# mlflow.set_tag("model_type", model_type)
# mlflow.set_tag("stage", "validation")
# mlflow.set_tag("threshold_strategy", "cost_optimized")

# mlflow.log_metric("auc", float(roc_auc_score(y_valid, y_proba)))
# mlflow.log_metric("f1", float(f1_score(y_valid, y_pred)))
# mlflow.log_metric("business_cost", float(threshold_info["cost"]))
# mlflow.log_metric("threshold", float(threshold_info["threshold"]))
# mlflow.log_metric("fit_time", float(fit_time))
# mlflow.log_metric("predict_time", float(predict_time))

# mlflow.log_metric("main_score", -float(threshold_info["cost"]))

track_run(
    run_name= run_name,
    model=best_model_lightGBM,
    X_train=X_train, y_train=y_train,
    X_valid=X_valid, y_valid=y_valid,
    params=gs.best_params_,
    extra_metrics=extra_metrics_lightGBM,
    y_valid_proba=y_proba,
    y_valid_pred=y_pred,
    threshold_info=threshold_info,
    fit_time=fit_time,
    predict_time=predict_time,
    cv_results_df=cv_results_df,
    notebook_run_id=notebook_run_id,
)

C:\apps\anaconda3\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/05/11 19:03:39 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


C:\apps\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\apps\anaconda3\Lib\site-packages\mlflow\tracking\_model_registry\utils.py:215: FutureWarning: The filesystem model registry backend (e.g., './mlruns') will be deprecated in February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://github.com/mlflow/mlflow/issues/18534 for more details and migration guidance.
  return FileStore(store_uri)
Successfully registered model 'lightgbm_LGBMClassifier'.
Created version '1' of model 'lightgbm_LGBMClassifier'.


### SAVE model to .joblib

In [9]:
import joblib

best_model = best_model_lightGBM #gs.best_estimator_

artifact = {
    "model": best_model,
    "threshold": threshold_info["threshold"],
    "threshold_info": threshold_info,
    "score_name": "business_cost_opt",
}

# joblib.dump(best_model, "../model/model.joblib")
joblib.dump(artifact, "../model/best_model_lightGBM.joblib")

print("✅ Model saved to model.joblib")
print("Best params:", gs.best_params_)
print("Best score:", gs.best_score_)

✅ Model saved to model.joblib
Best params: {'model__colsample_bytree': 0.8, 'model__learning_rate': 0.05, 'model__max_depth': 10, 'model__min_child_samples': 60, 'model__n_estimators': 400, 'model__num_leaves': 31, 'model__subsample': 0.8}
Best score: -21595.0


## 4) Baseline DummyClassifier (référence)

In [10]:
pipe, param_grid = gridsearch_dummy(X_train, y_train, cv)
t0 = time.time()
gs = run_gridsearch(pipe, param_grid, X_train, y_train, cv)
fit_time = time.time() - t0

best_model_dummyClassif = gs.best_estimator_
t1 = time.time()
y_proba = get_proba(best_model_dummyClassif, X_valid)
predict_time = time.time() - t1

threshold_info = optimal_threshold_cost(y_valid.values, y_proba)
y_pred = (y_proba >= threshold_info["threshold"]).astype(int)

# CV results artifact (utile pour stabilité & soutenance)
cv_results_df = pd.DataFrame(gs.cv_results_)

extra_metrics_dummyClassif = {
    "business_cost_test": threshold_info["cost"],
    "business_threshold_test": threshold_info["threshold"],
    "business_score_cv": gs.best_score_,
    "AUC_test": roc_auc_score(y_valid, y_proba),
    "mean_cv_threshold": gs.cv_results_["mean_test_threshold"][gs.best_index_],
    "fit_time": fit_time,
    "predict_time": predict_time  
}


Fitting 2 folds for each of 1 candidates, totalling 2 fits


### Résultat

In [11]:
gs.best_params_, gs.best_score_, extra_metrics_dummyClassif

({},
 np.float64(-32370.0),
 {'business_cost_test': 16190.0,
  'business_threshold_test': 0.05,
  'business_score_cv': np.float64(-32370.0),
  'AUC_test': 0.5,
  'mean_cv_threshold': np.float64(0.05),
  'fit_time': 5.628400087356567,
  'predict_time': 0.17236781120300293})

### MLFlow tracking

In [12]:
run_name = "dummy_baseline"

# model_type = f"{run_name}_threshold_cost_optimized"

# mlflow.set_tag("algo", run_name)
# mlflow.set_tag("model_type", model_type)
# mlflow.set_tag("stage", "validation")
# mlflow.set_tag("threshold_strategy", "cost_optimized")

# mlflow.log_metric("auc", float(roc_auc_score(y_valid, y_proba)))
# mlflow.log_metric("f1", float(f1_score(y_valid, y_pred)))
# mlflow.log_metric("business_cost", float(threshold_info["cost"]))
# mlflow.log_metric("threshold", float(threshold_info["threshold"]))
# mlflow.log_metric("fit_time", float(fit_time))
# mlflow.log_metric("predict_time", float(predict_time))

# mlflow.log_metric("main_score", -float(threshold_info["cost"]))

track_run(
    run_name=run_name,
    model=best_model_dummyClassif,
    X_train=X_train, y_train=y_train,
    X_valid=X_valid, y_valid=y_valid,
    params=gs.best_params_,
    extra_metrics=extra_metrics_dummyClassif,
    y_valid_proba=y_proba,
    y_valid_pred=y_pred,
    threshold_info=threshold_info,
    fit_time=fit_time,
    predict_time=predict_time,
    cv_results_df=cv_results_df,
    notebook_run_id=notebook_run_id,
)

C:\apps\anaconda3\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/05/11 19:04:05 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Successfully registered model 'dummy_baseline_DummyClassifier'.
Created version '1' of model 'dummy_baseline_DummyClassifier'.


## 5) XGBoost 

In [13]:
pipe, param_grid = gridsearch_xgb_smote(X_train, y_train, cv)
t0 = time.time()
gs = run_gridsearch(pipe, param_grid, X_train, y_train, cv)
fit_time = time.time() - t0

best_model_xgBoost = gs.best_estimator_
t1 = time.time()
y_proba = get_proba(best_model_xgBoost, X_valid)
predict_time = time.time() - t1

threshold_info = optimal_threshold_cost(y_valid.values, y_proba)
y_pred = (y_proba >= threshold_info["threshold"]).astype(int)

cv_results_df = pd.DataFrame(gs.cv_results_)

extra_metrics_xgBoost = {
    "business_cost_test": threshold_info["cost"],
    "business_threshold_test": threshold_info["threshold"],
    "business_score_cv": gs.best_score_,
    "AUC_test": roc_auc_score(y_valid, y_proba),
    "mean_cv_threshold": gs.cv_results_["mean_test_threshold"][gs.best_index_],
    "fit_time": fit_time,
    "predict_time": predict_time    
}

Fitting 2 folds for each of 2 candidates, totalling 4 fits


### Résultat

In [14]:
gs.best_params_, gs.best_score_, extra_metrics_xgBoost

({'model__colsample_bytree': 0.8,
  'model__learning_rate': 0.05,
  'model__max_depth': 4,
  'model__min_child_weight': 5,
  'model__n_estimators': 500,
  'model__subsample': 0.8},
 np.float64(-21790.5),
 {'business_cost_test': 10349.0,
  'business_threshold_test': 0.105,
  'business_score_cv': np.float64(-21790.5),
  'AUC_test': 0.7583113037440741,
  'mean_cv_threshold': np.float64(0.0925),
  'fit_time': 83.18273973464966,
  'predict_time': 0.22596311569213867})

### MLFlow tracking

In [15]:
run_name = "xgboost"

# model_type = f"{run_name}_threshold_cost_optimized"

# mlflow.set_tag("algo", run_name)
# mlflow.set_tag("model_type", model_type)
# mlflow.set_tag("stage", "validation")
# mlflow.set_tag("threshold_strategy", "cost_optimized")

# mlflow.log_metric("auc", float(roc_auc_score(y_valid, y_proba)))
# mlflow.log_metric("f1", float(f1_score(y_valid, y_pred)))
# mlflow.log_metric("business_cost", float(threshold_info["cost"]))
# mlflow.log_metric("threshold", float(threshold_info["threshold"]))
# mlflow.log_metric("fit_time", float(fit_time))
# mlflow.log_metric("predict_time", float(predict_time))

# mlflow.log_metric("main_score", -float(threshold_info["cost"]))

track_run(
    run_name=run_name,
    model=best_model_xgBoost,
    X_train=X_train, y_train=y_train,
    X_valid=X_valid, y_valid=y_valid,
    params=gs.best_params_,
    extra_metrics=extra_metrics_xgBoost,
    y_valid_proba=y_proba,
    y_valid_pred=y_pred,
    threshold_info=threshold_info,
    fit_time=fit_time,
    predict_time=predict_time,
    cv_results_df=cv_results_df,
    notebook_run_id=notebook_run_id,
)


C:\apps\anaconda3\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/05/11 19:05:40 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Successfully registered model 'xgboost_XGBClassifier'.
Created version '1' of model 'xgboost_XGBClassifier'.


### SAVE model to .joblib

In [16]:
import joblib

best_model = best_model_xgBoost #gs.best_estimator_

artifact = {
    "model": best_model,
    "threshold": threshold_info["threshold"],
    "threshold_info": threshold_info,
    "score_name": "business_cost_opt",
}

# joblib.dump(best_model, "../model/model.joblib")
joblib.dump(artifact, "../model/best_model_xgBoost.joblib")

print("✅ Model saved to model.joblib")
print("Best params:", gs.best_params_)
print("Best score:", gs.best_score_)

✅ Model saved to model.joblib
Best params: {'model__colsample_bytree': 0.8, 'model__learning_rate': 0.05, 'model__max_depth': 4, 'model__min_child_weight': 5, 'model__n_estimators': 500, 'model__subsample': 0.8}
Best score: -21790.5


In [17]:
# !pip freeze > requirements.txt

In [18]:
import sklearn
print(sklearn.__version__)

1.7.2


In [19]:
import imblearn
print(imblearn.__version__)

0.14.0


## 6) Logistic Regression + SMOTE

In [20]:
pipe, param_grid = gridsearch_logreg_smote(X_train, y_train, cv)
t0 = time.time()
gs = run_gridsearch(pipe, param_grid, X_train, y_train, cv)
fit_time = time.time() - t0

best_model_logRegression = gs.best_estimator_
t1 = time.time()
y_proba = get_proba(best_model_logRegression, X_valid)
predict_time = time.time() - t1

threshold_info = optimal_threshold_cost(y_valid.values, y_proba)
y_pred = (y_proba >= threshold_info["threshold"]).astype(int)

cv_results_df = pd.DataFrame(gs.cv_results_)

extra_metrics_logRegression = {
    "business_cost_test": threshold_info["cost"],
    "business_threshold_test": threshold_info["threshold"],
    "business_score_cv": gs.best_score_,
    "AUC_test": roc_auc_score(y_valid, y_proba),
    "mean_cv_threshold": gs.cv_results_["mean_test_threshold"][gs.best_index_],
    "fit_time": fit_time,
    "predict_time": predict_time    
}

Fitting 2 folds for each of 2 candidates, totalling 4 fits


C:\apps\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


### Résultats

In [21]:
gs.best_params_, gs.best_score_, extra_metrics_logRegression

({'model__C': 1, 'model__solver': 'lbfgs'},
 np.float64(-27455.0),
 {'business_cost_test': 13714.0,
  'business_threshold_test': 0.5299999999999999,
  'business_score_cv': np.float64(-27455.0),
  'AUC_test': 0.6348064855621551,
  'mean_cv_threshold': np.float64(0.5125),
  'fit_time': 287.64221811294556,
  'predict_time': 0.12801122665405273})

### MLFlow tracking

In [22]:
run_name = "logreg_smote"

# model_type = f"{run_name}_threshold_cost_optimized"

# mlflow.set_tag("algo", run_name)
# mlflow.set_tag("model_type", model_type)
# mlflow.set_tag("stage", "validation")
# mlflow.set_tag("threshold_strategy", "cost_optimized")

# mlflow.log_metric("auc", float(roc_auc_score(y_valid, y_proba)))
# mlflow.log_metric("f1", float(f1_score(y_valid, y_pred)))
# mlflow.log_metric("business_cost", float(threshold_info["cost"]))
# mlflow.log_metric("threshold", float(threshold_info["threshold"]))
# mlflow.log_metric("fit_time", float(fit_time))
# mlflow.log_metric("predict_time", float(predict_time))

# mlflow.log_metric("main_score", -float(threshold_info["cost"]))

track_run(
    run_name=run_name,
    model=best_model_logRegression,
    X_train=X_train, y_train=y_train,
    X_valid=X_valid, y_valid=y_valid,
    params=gs.best_params_,
    extra_metrics=extra_metrics_logRegression,
    y_valid_proba=y_proba,
    y_valid_pred=y_pred,
    threshold_info=threshold_info,
    fit_time=fit_time,
    predict_time=predict_time,
    cv_results_df=cv_results_df,
    notebook_run_id=notebook_run_id,
)

C:\apps\anaconda3\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/05/11 19:10:40 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Successfully registered model 'logreg_smote_LogisticRegression'.
Created version '1' of model 'logreg_smote_LogisticRegression'.


In [23]:
import mlflow

print("tracking uri =", mlflow.get_tracking_uri())

exp = mlflow.get_experiment_by_name("credit_scoring")
print("credit_scoring experiment =", exp)

runs = mlflow.search_runs()
print("Nb runs =", len(runs))
print(runs[["run_id", "experiment_id", "tags.mlflow.runName"]].tail(20))

tracking uri = file:./mlruns
credit_scoring experiment = <Experiment: artifact_location='file:C:/projects/openclassrooms_DS/P7/ds_project_7_credit_scoring/notebooks/mlruns/429337015077246251', creation_time=1778518788607, experiment_id='429337015077246251', last_update_time=1778518788607, lifecycle_stage='active', name='credit_scoring', tags={}>
Nb runs = 4
                             run_id       experiment_id tags.mlflow.runName
0  c71d53eb278648ee9705c210f5418464  429337015077246251        logreg_smote
1  737d72c2a12d401094b3c970e1d1ef16  429337015077246251             xgboost
2  825c9ca2ece94b14810035b207106ace  429337015077246251      dummy_baseline
3  b8854bedb065423cbdf6ff322fd254a7  429337015077246251            lightgbm


## 7) RandomForest + SMOTE

In [ ]:
pipe, param_grid = gridsearch_rf_smote(X_train, y_train, cv)
t0 = time.time()
gs = run_gridsearch(pipe, param_grid, X_train, y_train, cv)
fit_time = time.time() - t0

best_model_randomForest = gs.best_estimator_
t1 = time.time()
y_proba = get_proba(best_model_randomForest, X_valid)
predict_time = time.time() - t1

threshold_info = optimal_threshold_cost(y_valid.values, y_proba)
y_pred = (y_proba >= threshold_info["threshold"]).astype(int)

cv_results_df = pd.DataFrame(gs.cv_results_)

extra_metrics_randomForest = {
    "business_cost_test": threshold_info["cost"],
    "business_threshold_test": threshold_info["threshold"],
    "business_score_cv": gs.best_score_,
    "AUC_test": roc_auc_score(y_valid, y_proba),
    "mean_cv_threshold": gs.cv_results_["mean_test_threshold"][gs.best_index_],
    "fit_time": fit_time,
    "predict_time": predict_time
}

Fitting 2 folds for each of 1 candidates, totalling 2 fits


### Résultats

In [ ]:
gs.best_params_, gs.best_score_, extra_metrics_randomForest

### MLFlow tracking

In [ ]:
run_name = "rf_smote"

# model_type = f"{run_name}_threshold_cost_optimized"

# mlflow.set_tag("algo", run_name)
# mlflow.set_tag("model_type", model_type)
# mlflow.set_tag("stage", "validation")
# mlflow.set_tag("threshold_strategy", "cost_optimized")

# mlflow.log_metric("auc", float(roc_auc_score(y_valid, y_proba)))
# mlflow.log_metric("f1", float(f1_score(y_valid, y_pred)))
# mlflow.log_metric("business_cost", float(threshold_info["cost"]))
# mlflow.log_metric("threshold", float(threshold_info["threshold"]))
# mlflow.log_metric("fit_time", float(fit_time))
# mlflow.log_metric("predict_time", float(predict_time))

# mlflow.log_metric("main_score", -float(threshold_info["cost"]))

track_run(
    run_name=run_name,
    model=best_model_randomForest,
    X_train=X_train, y_train=y_train,
    X_valid=X_valid, y_valid=y_valid,
    params=gs.best_params_,
    extra_metrics=extra_metrics_randomForest,
    y_valid_proba=y_proba,
    y_valid_pred=y_pred,
    threshold_info=threshold_info,
    fit_time=fit_time,
    predict_time=predict_time,
    cv_results_df=cv_results_df,
    notebook_run_id=notebook_run_id,
)

## 8) À faire

- Ajouter d'autres algos (GradientBoosting, XGBoost/LightGBM si autorisé)
- Faire un **fine-tuning** autour des meilleurs hyperparamètres
- Comparer via MLflow UI (runs)

# Dana Tests

## Étape 1 — Comparer les algorithmes (simple)
### 1 algorithme = 1 run MLflow

In [ ]:
# import mlflow
# models = {
#     "LogisticRegression": best_model_logRegression,
#     "RandomForest": best_model_randomForest,
# }

# for name, model in models.items():
#     model.fit(X_train, y_train)
#     auc = roc_auc_score(y_valid, model.predict_proba(X_valid)[:, 1])

#     with mlflow.start_run(run_name=name):
#         mlflow.log_metric("auc", auc)
#         mlflow.log_param("model", name)


## Étape 2 — Affiner le meilleur (toujours simple)
### Supposons que RandomForest gagne.
### 1 run = 1 réglage

In [ ]:
# from sklearn.ensemble import RandomForestClassifier
# depths = [5, 10, 15]

# for d in depths:
#     rf = RandomForestClassifier(max_depth=d)
#     rf.fit(X_train, y_train)
#     auc = roc_auc_score(y_valid, rf.predict_proba(X_valid)[:, 1])

#     with mlflow.start_run(run_name=f"RF_depth_{d}"):
#         mlflow.log_metric("auc", auc)
#         mlflow.log_param("max_depth", d)
